# Разметка реальных документов через 7B (yurteg)
Прогоняем labeled_data_normalized.jsonl через дообученную 7B с CoT.
Заменяем GLM-метки на метки учителя. Fallback — оригинальная метка.

In [ ]:
# Пути — поправь если загрузил в другое место
GGUF_PATH = "/kaggle/input/yurteg-7b/qwen2.5-7b-instruct.Q4_K_M.gguf"
DATASET_PATH = "/kaggle/input/yurteg-dataset/labeled_data_normalized.jsonl"
OUTPUT_PATH = "/kaggle/working/labeled_data_7b.jsonl"

In [ ]:
!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python --quiet --no-cache-dir
!pip install tqdm --quiet

In [ ]:
from llama_cpp import Llama

llm = Llama(
    model_path=GGUF_PATH,
    n_gpu_layers=-1,   # всё на GPU
    n_ctx=2048,
    verbose=False,
)
print("Модель загружена")

In [ ]:
SYSTEM_PROMPT = """Ты — опытный юрист-аналитик. Извлеки структурированные метаданные из юридического документа.

ПРАВИЛА:
1. Отвечай в формате: сначала РАССУЖДЕНИЕ, потом JSON.
2. РАССУЖДЕНИЕ: 2-3 предложения — почему этот тип, кто контрагент, что предмет.
3. JSON строго после строки МЕТАДАННЫЕ:
4. Отсутствующую информацию ставь null.
5. Даты строго YYYY-MM-DD.
6. Сумму с пробелами и валютой: 1 500 000 руб.
7. parties и special_conditions — всегда массивы.
8. confidence — число от 0.0 до 1.0."""

USER_TEMPLATE = """Извлеки метаданные из текста юридического документа.

Верни в формате:
РАССУЖДЕНИЕ: [2-3 предложения]
МЕТАДАННЫЕ:
{{"document_type": "...", "counterparty": "...", "subject": "...", "date_signed": "...", "date_start": "...", "date_end": "...", "amount": "...", "special_conditions": [...], "parties": [...], "confidence": 0.0, "is_template": false}}

Текст документа:
{text}"""

In [ ]:
import json

def parse_response(content: str) -> tuple[str, dict] | None:
    """Парсим ответ: РАССУЖДЕНИЕ + МЕТАДАННЫЕ."""
    reasoning = ""
    try:
        if "РАССУЖДЕНИЕ:" in content and "МЕТАДАННЫЕ:" in content:
            parts = content.split("МЕТАДАННЫЕ:")
            reasoning = parts[0].replace("РАССУЖДЕНИЕ:", "").strip()
            json_part = parts[1].strip()
        elif "МЕТАДАННЫЕ:" in content:
            json_part = content.split("МЕТАДАННЫЕ:")[1].strip()
        else:
            json_part = content

        if json_part.startswith("```"):
            json_part = json_part.split("```")[1]
            if json_part.startswith("json"):
                json_part = json_part[4:]

        # Берём только первый JSON-объект
        brace = json_part.find("{")
        if brace != -1:
            json_part = json_part[brace:]

        metadata = json.loads(json_part.strip())

        # Чистим метаданные
        if "is_template" not in metadata:
            metadata["is_template"] = False
        if not isinstance(metadata.get("parties"), list):
            metadata["parties"] = []
        if not isinstance(metadata.get("special_conditions"), list):
            metadata["special_conditions"] = []
        for f in ["counterparty", "subject", "amount", "date_signed", "date_start", "date_end"]:
            if metadata.get(f) == "":
                metadata[f] = None
        for f in ["date_signed", "date_start", "date_end"]:
            val = metadata.get(f)
            if val and ("00-00" in str(val) or len(str(val)) != 10):
                metadata[f] = None

        return reasoning, metadata
    except Exception:
        return None


def extract(text: str) -> tuple[str, dict] | None:
    """Запускаем инференс 7B с CoT."""
    prompt = USER_TEMPLATE.format(text=text[:3000])
    resp = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        temperature=0,
        max_tokens=800,
    )
    content = resp["choices"][0]["message"]["content"].strip()
    return parse_response(content)

In [ ]:
from tqdm import tqdm
from datetime import datetime

records = []
with open(DATASET_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))
print(f"Загружено записей: {len(records)}")

In [ ]:
replaced = 0
kept_original = 0
errors = 0

with open(OUTPUT_PATH, "w", encoding="utf-8") as f_out:
    for record in tqdm(records, desc="Разметка", unit="doc"):
        input_text = record.get("input_text", "").strip()

        # Получаем оригинальную метку (fallback)
        orig_output = record.get("output", {})
        if isinstance(orig_output, str):
            try:
                orig_output = json.loads(orig_output)
            except Exception:
                orig_output = {}

        new_output = orig_output
        cot_reasoning = ""
        source = "glm_original"

        if input_text:
            try:
                result = extract(input_text)
                if result is not None:
                    reasoning, metadata = result
                    # Заменяем только если тип не пустой
                    if metadata.get("document_type"):
                        new_output = metadata
                        cot_reasoning = reasoning
                        source = "yurteg_7b"
                        replaced += 1
                    else:
                        kept_original += 1
                else:
                    kept_original += 1
            except Exception as e:
                errors += 1
                kept_original += 1

        out_record = {
            **record,
            "output": new_output,
            "cot_reasoning": cot_reasoning,
            "label_source": source,
            "relabeled_at": datetime.now().isoformat(),
        }
        f_out.write(json.dumps(out_record, ensure_ascii=False) + "\n")

print(f"\nГотово!")
print(f"  Заменено (7B):        {replaced}")
print(f"  Оставлено (оригинал): {kept_original}")
print(f"  Ошибок:               {errors}")
print(f"  Файл: {OUTPUT_PATH}")